# 추가 진단: c_gas, c_ics가 0으로 가는 이유

**핵심 발견**: c_gas ≈ 0.001, c_ics ≈ 0.003 (1000배 작음)

**가설**: GDE component map (gtmodel 출력)이 데이터 카운트의 ~1000배 큰 값을 가짐
→ Pi0/Bremss/ICS의 MapCubeFunction이 ~1000배 부풀려진 flux를 갖고 있음

이 진단들로 absolute scale을 검증합니다.

In [ ]:
# %% ══════════════════════════════════════════════════════════
# 진단 A: GDE component map vs CCUBE counts 직접 비교
# ══════════════════════════════════════════════════════════════

import numpy as np
import os
from astropy.io import fits

S = OFFSET
E = S + NX_FIT

# CCUBE counts (관측 데이터)
with fits.open(ccube_file) as h:
    counts = h[0].data[:, S:E, S:E]

# Convolved component maps (Model X)
model_name = 'X'
comps_data = {}
for comp in ['pion', 'bremss', 'ics', 'GCE', 'Fermi_bubble', 'isotropic']:
    fpath = os.path.join(COMPMAP_DIR, f'{comp}_model{model_name}.fits')
    if os.path.exists(fpath):
        with fits.open(fpath) as h:
            comps_data[comp] = h[0].data[:, S:E, S:E]

print('=== Total counts/expected per energy bin (40x40 ROI, no mask) ===')
print(f'{"ie":>3} {"E[GeV]":>8} {"obs":>12} {"pi0+brm":>12} {"ics":>12} {"GCE":>12} {"bub":>12} {"iso":>12} {"sum_model":>12} {"obs/sum":>10}')
print('-' * 110)
for ie in range(N_ENERGY_BINS):
    obs_ie = counts[ie].sum()
    pi0_ie = comps_data.get('pion', np.zeros_like(counts))[ie].sum()
    brm_ie = comps_data.get('bremss', np.zeros_like(counts))[ie].sum()
    ics_ie = comps_data.get('ics', np.zeros_like(counts))[ie].sum()
    gce_ie = comps_data.get('GCE', np.zeros_like(counts))[ie].sum()
    bub_ie = comps_data.get('Fermi_bubble', np.zeros_like(counts))[ie].sum()
    iso_ie = comps_data.get('isotropic', np.zeros_like(counts))[ie].sum()
    sum_ie = pi0_ie + brm_ie + ics_ie + gce_ie + bub_ie + iso_ie
    ratio = obs_ie / sum_ie if sum_ie > 0 else 0
    print(f'{ie:3d} {ENERGY_CENTERS_GEV[ie]:8.3f} {obs_ie:12.0f} '
          f'{pi0_ie+brm_ie:12.0f} {ics_ie:12.0f} {gce_ie:12.2e} '
          f'{bub_ie:12.2e} {iso_ie:12.2e} {sum_ie:12.0f} {ratio:10.4f}')

print('\n해석:')
print('  - obs/sum이 ~1.0이면: gtmodel output이 데이터와 같은 단위 (정상)')
print('  - obs/sum이 ~0.001이면: gtmodel이 데이터의 1000배 큰 값 출력')
print('  - 만약 GCE column이 0이면: GCE template normalization이 (1e-11)로 너무 작음')

In [ ]:
# %% ══════════════════════════════════════════════════════════
# 진단 B: MapCube 파일의 absolute flux 확인
# ══════════════════════════════════════════════════════════════

mapcube_file = os.path.join(MAPCUBE_DIR, 'pi0_mapcube_modelX.fits')
print(f'pi0 MapCube 파일: {mapcube_file}')

with fits.open(mapcube_file) as h:
    cube = h[0].data
    hdr = h[0].header
    print(f'  shape: {cube.shape}')
    print(f'  CDELT1 (lon pix size): {hdr.get("CDELT1")} deg')
    print(f'  CDELT2 (lat pix size): {hdr.get("CDELT2")} deg')
    print(f'  CTYPE3: {hdr.get("CTYPE3")}')
    print(f'  CUNIT3: {hdr.get("CUNIT3", "NOT SET")}')
    
    if 'ENERGIES' in [hdu.name for hdu in h]:
        energies = h['ENERGIES'].data['Energy']
        print(f'\n  ENERGIES table (n={len(energies)}):')
        print(f'    range: [{energies.min():.3f}, {energies.max():.3f}]')
        print(f'    이 값들이 어떤 단위인지 확인 필요')
        if energies.max() > 1000:
            print(f'    → MeV 단위로 추정 (max > 1000)')
        else:
            print(f'    → GeV 단위로 추정 (max < 1000)')
        print(f'    Fermi 표준은 MeV. 만약 GeV로 저장했다면 ~1000배 차이!')

print(f'\n=== Pixel value 검증 (1 GeV 근처) ===')
# 1 GeV 근처의 빈 찾기
with fits.open(mapcube_file) as h:
    energies = h['ENERGIES'].data['Energy']
    if energies.max() > 1000:  # MeV
        target_e = 1000.0  # 1 GeV in MeV
    else:  # GeV
        target_e = 1.0
    ie_1gev = int(np.argmin(np.abs(energies - target_e)))
    print(f'1 GeV 근처 bin: ie={ie_1gev}, E={energies[ie_1gev]}')
    
    cube_slice = h[0].data[ie_1gev]
    print(f'  shape: {cube_slice.shape}')
    # GC 픽셀 (300, 300)
    gc_val = cube_slice[300, 300]
    print(f'  값 @ GC pixel: {gc_val:.3e}')
    print(f'  최대값: {cube_slice.max():.3e}')
    print(f'  단위: ph/cm²/s/MeV/sr (Fermi MapCubeFunction 표준)')
    print(f'\n  비교: 1 GeV 근처 Galactic plane Pi0 flux는 보통 ~1e-7 ph/cm²/s/MeV/sr')
    print(f'  만약 ~1e-4면 1000배 큰 것 (단위 변환 실수)')
    print(f'  만약 ~1e-7이면 정상, 다른 곳에 문제')

In [ ]:
# %% ══════════════════════════════════════════════════════════
# 진단 C: Sanghwan의 reference Mapcube 비교 (있다면)
# ══════════════════════════════════════════════════════════════

# Sanghwan의 메인 노트북은 다음 경로에서 MapCube 사용:
# /home/sanghwan/FermiLAT/Mapcube_convert_test/pion_mapcube_modelX.gz
# 사용자 시스템에 있을 가능성이 있음

candidate_paths = [
    '/home/sanghwan/FermiLAT/Mapcube_convert_test/pion_mapcube_modelX.gz',
    '/home/haebarg/FermiLAT/Mapcube_convert_test/pion_mapcube_modelX.gz',
    '/home/haebarg/Mapcube_convert_test/pion_mapcube_modelX.gz',
    '/home/haebarg/GCE-Chi-square-fitting/Mapcube_convert_test/pion_mapcube_modelX.gz',
    '/home/haebarg/GCE-Chi-square-fitting/GCE_TEMPLATES_FILES_v3/pi0_X_Map_flux_AAfrag_AMS02_InnerGalaxy.fits',
]

print('Sanghwan의 reference MapCube 또는 Zenodo 원본 검색...')
for p in candidate_paths:
    if os.path.exists(p):
        print(f'\n✅ 발견: {p}')
        try:
            with fits.open(p) as h:
                if len(h) > 0 and h[0].data is not None:
                    cube = h[0].data
                    print(f'   shape: {cube.shape}')
                    print(f'   max: {cube.max():.3e}')
                    print(f'   center [N//2, N//2]: {cube[cube.shape[0]//2, cube.shape[-2]//2, cube.shape[-1]//2] if cube.ndim==3 else cube[cube.shape[-2]//2, cube.shape[-1]//2]:.3e}')
                if 'ENERGIES' in [hdu.name for hdu in h]:
                    en = h['ENERGIES'].data['Energy']
                    print(f'   ENERGIES: [{en.min():.3f}, {en.max():.3f}]')
        except Exception as e:
            print(f'   에러: {e}')
    else:
        print(f'  없음: {p}')

# Zenodo 원본 디렉토리 검색
print('\n\n=== Zenodo 원본 디렉토리 ===')
for guess in [
    '/home/haebarg/GCE-Chi-square-fitting/GCE_TEMPLATES_FILES_v3/',
    '/home/haebarg/Zenodo_GDE/',
    GDE_DIR if 'GDE_DIR' in dir() else None,
]:
    if guess and os.path.exists(guess):
        print(f'✅ {guess} 존재')
        # pi0 모델 X 파일 찾기
        import glob
        for pattern in ['pi0_X*.fits', 'pi0_*X*Map*.fits']:
            matches = glob.glob(os.path.join(guess, pattern))
            if matches:
                print(f'   {pattern}: {len(matches)} files')
                # 첫 파일 검사
                with fits.open(matches[0]) as h:
                    print(f'   첫 파일: {os.path.basename(matches[0])}')
                    print(f'      shape: {h[0].data.shape}')
                    print(f'      max: {h[0].data.max():.3e}')
                    print(f'      mean of nonzero: {h[0].data[h[0].data > 0].mean():.3e}')
                    if 'ENERGIES' in [hdu.name for hdu in h]:
                        en = h['ENERGIES'].data['Energy']
                        print(f'      ENERGIES: [{en.min():.3f}, {en.max():.3f}]')
                        print(f'      단위 추정: {"MeV" if en.max() > 1000 else "GeV"}')
                break